# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashar2005/Flyrank-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## 1. My lane

   **Lane 2: Refresh / Content Opportunity Scoring.**

   I'm choosing this lane because I've already run the starter pipeline end-to-end
   (Notebooks 01 and 02) and seen it work on real data: a learned model beat a
   hand-written rule by roughly 3x on Precision@50. That gave me a concrete feel
   for the problem before committing to it. I also want a capstone with a clear
   action at the end — a ranked list someone can actually act on — rather than
   pure exploratory analysis.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The question

**Decision:** Which pages should a content editor review first for refresh,
given limited review capacity?

**Unit of analysis:** A page (`content_id`) — one row per page, evaluated on
its most recent 90-day window of signals.

**Output:** A ranked queue of pages, ordered by refresh-review priority, with
reason codes explaining why each page was flagged (e.g. stale + visible,
declining with demand, page-one decay risk).

**Action:** An editor works down the ranked list and reviews/refreshes the
top candidates first, instead of guessing or working through pages in no
particular order.

**Cost of a wrong call:**
- False positive (flagged but not actually worth refreshing): wasted editor
  hours on a page that wasn't the real problem.
- False negative (a genuinely declining page ranked low or missed): the page
  keeps losing visibility/traffic while nobody looks at it — a slower,
  compounding cost.

**Why a plain rule isn't enough:** The starter pipeline already tested this.
A simple hand-written rule combining staleness, visibility, position, and
depth got Precision@50 = 0.240 (about 12 of the top 50 flagged pages were
actually declining). A random forest using the same underlying signals got
Precision@50 = 0.740 (about 37 of 50) — roughly 3x better. That gap shows the
signals interact in ways too tangled to hand-code with simple if-statements,
which is exactly where ML earns its place over a fixed rule.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [13]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ashar2005/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # from work/notebooks/ back to repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check the path"
print("Starter data found. You're ready.")

Working dir: /content/Flyrank-ML-Internship/Flyrank-ML-Internship
Starter data found. You're ready.


In [14]:
# Run the pipeline once so outputs/model_results.json exists
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/Flyrank-ML-Internship/Flyrank-ML-Internship/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/Flyrank-ML-Internship/Flyrank-ML-Internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/Flyrank-ML-Internship/Flyrank-ML-Internship/data/processed/model_predictions.csv
Wrote model results: /content/Flyrank-ML-Internship/Flyrank-ML-Internship/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /c

In [15]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Total pages:", df.shape[0])
print("Declining rate:", round((df["trend_direction"] == "down").mean(), 3))

# From the already-run starter pipeline (outputs/model_results.json):
import json
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"Hand-rule Precision@50: {base:.3f}")
print(f"Random forest Precision@50: {rf:.3f}  ({rf/base:.1f}x the hand rule)")

Total pages: 30000
Declining rate: 0.542
Hand-rule Precision@50: 0.240
Random forest Precision@50: 0.740  (3.1x the hand rule)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful words: what I can and can't claim

**What I can claim:** On this 30,000-row anonymized starter slice, using
client-holdout validation, a learned model produces a substantially better
top-50 review queue than a hand-written rule — this is an *observed,
directional* result on this dataset.

**What I can't claim:**
- I can't claim any recommendation, if acted on, will *cause* a page to
  recover — that requires a real experiment, not this data.
- The current label (`trend_direction == "down"`) is a proxy: it's a bucket
  computed from the *current* window, not a future outcome. A stronger
  capstone version of this should predict future decline/recovery over
  a forward window (e.g. prior 90 days → next 30 days), not the current
  state — I plan to move toward that once I start the warehouse data in
  Week 3+.
- I'm not claiming anything about Google's actual ranking algorithm.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.